# P10.6-AI — Notebook 63 v2: entrenamiento subarticular Axial T2

Flujo resistente a cortes y cambios de cuenta.

**Fase A — CPU:** ejecutar `1 → 2 → 3A`. La celda 3A guarda cada `.npy` completo directamente en Drive y puede retomarse.

**Fase B — GPU:** cambiar a T4 o superior y volver a ejecutar `1 → 2 → 3B → 4 → 5`. La celda 3B copia el caché completo al SSD local.

No ejecutar 3A desde dos runtimes al mismo tiempo. El `internal_test` permanece sellado para el Notebook 64.

`humanReviewRequired=true` · `notClinicalDiagnosis=true` · `officialTestAccessed=false`


In [1]:
# 1) Dependencias, Drive y pipeline
from __future__ import annotations

import getpass
import importlib.util
import json
import shutil
import subprocess
import sys
from pathlib import Path

import torch
from google.colab import drive  # type: ignore

packages = {
    "pydicom": "pydicom",
    "timm": "timm",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
    "kaggle": "kaggle",
}
missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", *missing
    ])

print({
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "torch": torch.__version__,
    "installedNow": missing,
})
drive.mount("/content/drive", force_remount=False)

REPO_URL = (
    "https://github.com/EnzoAA004/"
    "PFI_MVPTest_Enzo_AImodule.git"
)
REPO_REF = "enzo/p10-6-ai-rsna-findings"
REPO_ROOT = Path("/content/PFI_MVPTest_Enzo_AImodule")

if not (REPO_ROOT / ".git").exists():
    subprocess.check_call([
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        REPO_URL, str(REPO_ROOT),
    ])
else:
    subprocess.check_call(
        ["git", "fetch", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "checkout", REPO_REF],
        cwd=REPO_ROOT,
    )
    subprocess.check_call(
        ["git", "pull", "--ff-only", "origin", REPO_REF],
        cwd=REPO_ROOT,
    )

REPO_SHA = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_ROOT,
    text=True,
).strip()

ai_service_path = str(REPO_ROOT / "ai_service")
if ai_service_path not in sys.path:
    sys.path.insert(0, ai_service_path)

from pfi_ai_service.training.rsna_subarticular_training import (
    TrainConfig,
    build_cache,
    download_selected_series,
    find_data_root,
    load_manifests,
    prepare_samples,
    train_model,
)

print({"repoRef": REPO_REF, "repoSha": REPO_SHA})


{'device': 'cuda', 'gpu': 'NVIDIA L4', 'torch': '2.11.0+cu128', 'installedNow': ['pydicom']}
Mounted at /content/drive
{'repoRef': 'enzo/p10-6-ai-rsna-findings', 'repoSha': '62c40f528ff2349fea4c22301972547457683bed'}


In [2]:
# Fase GPU rápida:
# cargar manifests, copiar el TAR al SSD y preparar entrenamiento.

from pathlib import Path
import json
import shutil
import tarfile
import time

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Esta celda requiere una GPU."
    )

PFI_ROOT = Path(
    "/content/drive/MyDrive/PFI_MVP"
)

RESULTS_ROOT = (
    PFI_ROOT
    / "results"
    / "P10_6_rsna_findings"
)

SPLIT_ROOT = (
    RESULTS_ROOT
    / "notebook62_subarticular_split"
)

RUN_ROOT = (
    RESULTS_ROOT
    / "notebook63_subarticular_training"
)

MODEL_ROOT = (
    PFI_ROOT
    / "models"
    / "P10_6_rsna_findings"
    / "subarticular_axial_t2_2p5d"
)

CHECKPOINT_ROOT = (
    MODEL_ROOT
    / "checkpoints"
)

DRIVE_ARCHIVE = (
    PFI_ROOT
    / "cache"
    / "notebook63_subarticular_cache.tar"
)

LOCAL_ARCHIVE = Path(
    "/content/notebook63_subarticular_cache.tar"
)

LOCAL_CACHE_ROOT = Path(
    "/content/rsna_subarticular_cache"
)

CFG = TrainConfig(
    seed=2026,
    image_size=224,
    crop_size=256,
    batch_size=32,
    num_workers=2,
    max_epochs=15,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    model_name="efficientnet_b0",
    pretrained=True,
    minimum_macro_f1=0.36,
    minimum_balanced_accuracy=0.45,
    minimum_severe_recall=0.30,
    minimum_moderate_recall=0.25,
)

for path in (
    RUN_ROOT,
    MODEL_ROOT,
    CHECKPOINT_ROOT,
):
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

# No abre internal_test; carga solamente train y validation.
train_manifest, validation_manifest, split_summary, manifest_hashes = (
    load_manifests(SPLIT_ROOT)
)


def add_cache_file(frame):
    """
    Reconstruye exactamente el nombre usado por build_cache,
    sin volver a recorrer los DICOM desde Drive.
    """
    result = frame.copy()

    result["cache_file"] = result.apply(
        lambda row: (
            f"{row['study_id']}__"
            f"{row['coordinate_series_id']}__"
            f"{row['side']}__"
            f"{str(row['level']).replace('-', '_')}__"
            f"{int(row['coordinate_instance_number'])}.npy"
        ),
        axis=1,
    )

    if result["cache_file"].duplicated().any():
        raise RuntimeError(
            "Se detectaron nombres de caché duplicados."
        )

    return result.reset_index(drop=True)


train_samples = add_cache_file(
    train_manifest
)

validation_samples = add_cache_file(
    validation_manifest
)

expected_names = {
    f"train/{name}"
    for name in train_samples["cache_file"].astype(str)
}

expected_names.update({
    f"validation/{name}"
    for name in validation_samples["cache_file"].astype(str)
})

if not DRIVE_ARCHIVE.is_file():
    raise RuntimeError(
        f"No se encontró el TAR: {DRIVE_ARCHIVE}"
    )

archive_size = DRIVE_ARCHIVE.stat().st_size

print({
    "gpu": torch.cuda.get_device_name(0),
    "archiveGiB": round(
        archive_size / 1024**3,
        2,
    ),
    "expectedFiles": len(expected_names),
    "stage": "copyingArchiveToLocalSSD",
})

LOCAL_ARCHIVE.unlink(
    missing_ok=True
)

if LOCAL_CACHE_ROOT.exists():
    shutil.rmtree(
        LOCAL_CACHE_ROOT
    )

LOCAL_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

# Copiar el único TAR desde Drive con progreso.
copied = 0
report_every = 256 * 1024**2
next_report = report_every
started = time.time()

with (
    DRIVE_ARCHIVE.open("rb") as source,
    LOCAL_ARCHIVE.open("wb") as target,
):
    while True:
        chunk = source.read(
            64 * 1024**2
        )

        if not chunk:
            break

        target.write(chunk)
        copied += len(chunk)

        if (
            copied >= next_report
            or copied == archive_size
        ):
            print({
                "copiedGiB": round(
                    copied / 1024**3,
                    2,
                ),
                "totalGiB": round(
                    archive_size / 1024**3,
                    2,
                ),
                "percent": round(
                    100 * copied / archive_size,
                    2,
                ),
                "minutes": round(
                    (time.time() - started) / 60,
                    1,
                ),
            })

            next_report += report_every

if LOCAL_ARCHIVE.stat().st_size != archive_size:
    raise RuntimeError(
        "La copia local del TAR quedó incompleta."
    )

print({
    "stage": "validatingArchive",
})

with tarfile.open(
    LOCAL_ARCHIVE,
    mode="r",
) as archive:

    members = archive.getmembers()

    actual_names = {
        member.name
        for member in members
        if member.isfile()
    }

    missing = sorted(
        expected_names - actual_names
    )

    unexpected = sorted(
        actual_names - expected_names
    )

    if missing or unexpected:
        raise RuntimeError({
            "message": (
                "El contenido del TAR no coincide "
                "con los manifests."
            ),
            "missingCount": len(missing),
            "unexpectedCount": len(unexpected),
            "missingExamples": missing[:10],
            "unexpectedExamples": unexpected[:10],
        })

    print({
        "stage": "extractingToLocalSSD",
        "archiveFiles": len(actual_names),
    })

    archive.extractall(
        path=LOCAL_CACHE_ROOT,
        filter="data",
    )

local_train_names = {
    path.name
    for path in (
        LOCAL_CACHE_ROOT / "train"
    ).glob("*.npy")
}

local_validation_names = {
    path.name
    for path in (
        LOCAL_CACHE_ROOT / "validation"
    ).glob("*.npy")
}

expected_train_names = set(
    train_samples["cache_file"].astype(str)
)

expected_validation_names = set(
    validation_samples["cache_file"].astype(str)
)

if local_train_names != expected_train_names:
    raise RuntimeError(
        "El caché local de train no coincide."
    )

if (
    local_validation_names
    != expected_validation_names
):
    raise RuntimeError(
        "El caché local de validation no coincide."
    )

CACHE_ROOT = LOCAL_CACHE_ROOT

gpu_report = {
    "gpu": torch.cuda.get_device_name(0),
    "trainingCacheRoot": str(CACHE_ROOT),
    "trainFiles": len(local_train_names),
    "validationFiles": len(local_validation_names),
    "archiveFiles": len(actual_names),
    "internalTestAccessed": False,
    "readyForTraining": True,
}

print(json.dumps(
    gpu_report,
    indent=2,
    ensure_ascii=False,
))

{'gpu': 'NVIDIA L4', 'archiveGiB': 2.32, 'expectedFiles': 16339, 'stage': 'copyingArchiveToLocalSSD'}
{'copiedGiB': 0.25, 'totalGiB': 2.32, 'percent': 10.77, 'minutes': 0.1}
{'copiedGiB': 0.5, 'totalGiB': 2.32, 'percent': 21.54, 'minutes': 0.2}
{'copiedGiB': 0.75, 'totalGiB': 2.32, 'percent': 32.3, 'minutes': 0.3}
{'copiedGiB': 1.0, 'totalGiB': 2.32, 'percent': 43.07, 'minutes': 0.4}
{'copiedGiB': 1.25, 'totalGiB': 2.32, 'percent': 53.84, 'minutes': 0.6}
{'copiedGiB': 1.5, 'totalGiB': 2.32, 'percent': 64.61, 'minutes': 0.7}
{'copiedGiB': 1.75, 'totalGiB': 2.32, 'percent': 75.37, 'minutes': 0.8}
{'copiedGiB': 2.0, 'totalGiB': 2.32, 'percent': 86.14, 'minutes': 0.9}
{'copiedGiB': 2.25, 'totalGiB': 2.32, 'percent': 96.91, 'minutes': 1.0}
{'copiedGiB': 2.32, 'totalGiB': 2.32, 'percent': 100.0, 'minutes': 1.0}
{'stage': 'validatingArchive'}
{'stage': 'extractingToLocalSSD', 'archiveFiles': 16339}
{
  "gpu": "NVIDIA L4",
  "trainingCacheRoot": "/content/rsna_subarticular_cache",
  "trainFile

In [2]:

# 2) Rutas, configuración, manifests y DICOM
PFI_ROOT = Path("/content/drive/MyDrive/PFI_MVP")
if not PFI_ROOT.is_dir():
    raise RuntimeError(
        "No se encontró PFI_MVP en Mi unidad. "
        "Agregá la carpeta compartida como acceso directo o corregí PFI_ROOT."
    )

RESULTS_ROOT = PFI_ROOT / "results" / "P10_6_rsna_findings"
SPLIT_ROOT = RESULTS_ROOT / "notebook62_subarticular_split"
RUN_ROOT = RESULTS_ROOT / "notebook63_subarticular_training"
MODEL_ROOT = (
    PFI_ROOT / "models" / "P10_6_rsna_findings"
    / "subarticular_axial_t2_2p5d"
)
CHECKPOINT_ROOT = MODEL_ROOT / "checkpoints"

LOCAL_DATA_ROOT = Path("/content/RSNA_LUMBAR_DISC")
DRIVE_DATA_ROOT = PFI_ROOT / "data" / "RSNA_LUMBAR_DISC"
PERSISTENT_CACHE_ROOT = (
    PFI_ROOT / "cache" / "notebook63_subarticular_cache"
)
LOCAL_CACHE_ROOT = Path("/content/rsna_subarticular_cache")
COMPETITION = "rsna-2024-lumbar-spine-degenerative-classification"

CFG = TrainConfig(
    seed=2026,
    image_size=224,
    crop_size=256,
    batch_size=32,
    num_workers=2,
    max_epochs=15,
    patience=5,
    learning_rate=2e-4,
    weight_decay=1e-4,
    model_name="efficientnet_b0",
    pretrained=True,
    minimum_macro_f1=0.36,
    minimum_balanced_accuracy=0.45,
    minimum_severe_recall=0.30,
    minimum_moderate_recall=0.25,
)

for path in (RUN_ROOT, MODEL_ROOT, CHECKPOINT_ROOT, PERSISTENT_CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)

train_manifest, validation_manifest, split_summary, manifest_hashes = (
    load_manifests(SPLIT_ROOT)
)

data_root, data_audits = find_data_root(
    [LOCAL_DATA_ROOT, DRIVE_DATA_ROOT],
    train_manifest,
    validation_manifest,
)
print({
    "trainRows": len(train_manifest),
    "validationRows": len(validation_manifest),
    "dataAudits": [
        {
            "root": audit.root,
            "complete": audit.complete,
            "missingSeries": audit.missing_series,
        }
        for audit in data_audits
    ],
    "internalTestAccessed": False,
})

if data_root is None:
    kaggle_token = getpass.getpass("Kaggle API token: ")
    data_root = download_selected_series(
        train_manifest,
        validation_manifest,
        LOCAL_DATA_ROOT,
        COMPETITION,
        kaggle_token,
    )

print({
    "selectedDataRoot": str(data_root),
    "persistentCacheRoot": str(PERSISTENT_CACHE_ROOT),
    "checkpointRoot": str(CHECKPOINT_ROOT),
})


{'trainRows': 13445, 'validationRows': 2894, 'dataAudits': [{'root': '/content/RSNA_LUMBAR_DISC', 'complete': False, 'missingSeries': 1990}, {'root': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'complete': True, 'missingSeries': 0}], 'internalTestAccessed': False}
{'selectedDataRoot': '/content/drive/MyDrive/PFI_MVP/data/RSNA_LUMBAR_DISC', 'persistentCacheRoot': '/content/drive/MyDrive/PFI_MVP/cache/notebook63_subarticular_cache', 'checkpointRoot': '/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings/subarticular_axial_t2_2p5d/checkpoints'}


## 3A — CPU: construir o reanudar caché persistente

Esta celda reutiliza los `.npy` existentes. Para pausar: detener manualmente 3A y ejecutar la celda **3A-checkpoint**.


In [3]:
# 3A) Construir o reanudar caché persistente
train_samples = prepare_samples(train_manifest, data_root, "train")
validation_samples = prepare_samples(
    validation_manifest,
    data_root,
    "validation",
)

for temporary in PERSISTENT_CACHE_ROOT.rglob("*.tmp.npy"):
    temporary.unlink(missing_ok=True)

train_cache_audit = build_cache(
    train_samples,
    PERSISTENT_CACHE_ROOT,
    "train",
    CFG,
)
validation_cache_audit = build_cache(
    validation_samples,
    PERSISTENT_CACHE_ROOT,
    "validation",
    CFG,
)

cache_report = {
    "cacheRoot": str(PERSISTENT_CACHE_ROOT),
    "train": {
        "present": len(list(
            (PERSISTENT_CACHE_ROOT / "train").glob("*.npy")
        )),
        "expected": len(train_samples),
    },
    "validation": {
        "present": len(list(
            (PERSISTENT_CACHE_ROOT / "validation").glob("*.npy")
        )),
        "expected": len(validation_samples),
    },
    "trainAudit": train_cache_audit,
    "validationAudit": validation_cache_audit,
    "internalTestAccessed": False,
}
cache_report["complete"] = (
    cache_report["train"]["present"] == cache_report["train"]["expected"]
    and cache_report["validation"]["present"]
    == cache_report["validation"]["expected"]
)

(RUN_ROOT / "persistent_cache_audit.json").write_text(
    json.dumps(cache_report, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print(json.dumps(cache_report, indent=2, ensure_ascii=False))

if not cache_report["complete"]:
    raise RuntimeError("Caché incompleto: volver a ejecutar 3A.")


cache train por serie:   0%|          | 0/1631 [00:00<?, ?it/s]

cache validation por serie:   0%|          | 0/359 [00:00<?, ?it/s]

{
  "cacheRoot": "/content/drive/MyDrive/PFI_MVP/cache/notebook63_subarticular_cache",
  "train": {
    "present": 13445,
    "expected": 13445
  },
  "validation": {
    "present": 2894,
    "expected": 2894
  },
  "trainAudit": {
    "split": "train",
    "expectedSamples": 13445,
    "builtSamples": 0,
    "reusedSamples": 13445,
    "cacheFiles": 13445,
    "minutes": 0.2
  },
  "validationAudit": {
    "split": "validation",
    "expectedSamples": 2894,
    "builtSamples": 0,
    "reusedSamples": 2894,
    "cacheFiles": 2894,
    "minutes": 0.04
  },
  "internalTestAccessed": false,
  "complete": true
}


In [4]:
# 3A-checkpoint) Ejecutar solo después de interrumpir 3A
import time

time.sleep(5)
temporary_files = list(PERSISTENT_CACHE_ROOT.rglob("*.tmp.npy"))
for temporary in temporary_files:
    temporary.unlink(missing_ok=True)

print(json.dumps({
    "cacheRoot": str(PERSISTENT_CACHE_ROOT),
    "persistent": str(PERSISTENT_CACHE_ROOT).startswith("/content/drive/"),
    "trainCached": len(list(
        (PERSISTENT_CACHE_ROOT / "train").glob("*.npy")
    )),
    "trainExpected": len(train_manifest),
    "validationCached": len(list(
        (PERSISTENT_CACHE_ROOT / "validation").glob("*.npy")
    )),
    "validationExpected": len(validation_manifest),
    "temporaryFilesRemoved": len(temporary_files),
    "safeToClose": True,
}, indent=2, ensure_ascii=False))


{
  "cacheRoot": "/content/drive/MyDrive/PFI_MVP/cache/notebook63_subarticular_cache",
  "persistent": true,
  "trainCached": 13445,
  "trainExpected": 13445,
  "validationCached": 2894,
  "validationExpected": 2894,
  "temporaryFilesRemoved": 0,
  "safeToClose": true
}


In [1]:
from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False,
)

Mounted at /content/drive


In [3]:
# 3A-archive) Crear un TAR robusto desde el caché persistente

from pathlib import Path
import json
import os
import shutil
import tarfile
import time

CACHE_SOURCE = Path(
    "/content/drive/MyDrive/PFI_MVP/cache/"
    "notebook63_subarticular_cache"
)

LOCAL_ARCHIVE = Path(
    "/content/notebook63_subarticular_cache.tar"
)

DRIVE_ARCHIVE = Path(
    "/content/drive/MyDrive/PFI_MVP/cache/"
    "notebook63_subarticular_cache.tar"
)

DRIVE_ARCHIVE_PARTIAL = Path(
    str(DRIVE_ARCHIVE) + ".partial"
)

train_files = sorted(
    (CACHE_SOURCE / "train").glob("*.npy")
)

validation_files = sorted(
    (CACHE_SOURCE / "validation").glob("*.npy")
)

all_files = train_files + validation_files

expected_names = {
    path.relative_to(CACHE_SOURCE).as_posix()
    for path in all_files
}

print({
    "train": len(train_files),
    "validation": len(validation_files),
    "total": len(all_files),
})

if len(train_files) != 13445:
    raise RuntimeError(
        f"Train incompleto: {len(train_files)}/13445"
    )

if len(validation_files) != 2894:
    raise RuntimeError(
        "Validation incompleto: "
        f"{len(validation_files)}/2894"
    )

free_gib = (
    shutil.disk_usage("/content").free
    / 1024**3
)

print({
    "localFreeGiB": round(free_gib, 2),
})

if free_gib < 5:
    raise RuntimeError(
        "No hay suficiente espacio libre en /content."
    )

# Eliminar solamente archivos TAR previos o parciales.
LOCAL_ARCHIVE.unlink(missing_ok=True)
DRIVE_ARCHIVE_PARTIAL.unlink(missing_ok=True)

started = time.time()

try:
    with tarfile.open(
        LOCAL_ARCHIVE,
        mode="w",
    ) as archive:

        for index, source_file in enumerate(
            all_files,
            start=1,
        ):
            relative_name = (
                source_file
                .relative_to(CACHE_SOURCE)
                .as_posix()
            )

            archive.add(
                source_file,
                arcname=relative_name,
                recursive=False,
            )

            if (
                index % 500 == 0
                or index == len(all_files)
            ):
                elapsed_minutes = (
                    time.time() - started
                ) / 60

                print({
                    "archived": index,
                    "total": len(all_files),
                    "percent": round(
                        100 * index / len(all_files),
                        2,
                    ),
                    "minutes": round(
                        elapsed_minutes,
                        1,
                    ),
                })

except Exception:
    LOCAL_ARCHIVE.unlink(missing_ok=True)
    raise

print({
    "stage": "validatingLocalArchive",
    "sizeGiB": round(
        LOCAL_ARCHIVE.stat().st_size
        / 1024**3,
        2,
    ),
})

# Validar que estén exactamente los archivos esperados.
with tarfile.open(
    LOCAL_ARCHIVE,
    mode="r",
) as archive:
    actual_names = {
        member.name
        for member in archive
        if (
            member.isfile()
            and member.name.endswith(".npy")
        )
    }

missing_names = sorted(
    expected_names - actual_names
)

unexpected_names = sorted(
    actual_names - expected_names
)

if missing_names or unexpected_names:
    raise RuntimeError({
        "message": "El TAR local no coincide con el caché.",
        "missingCount": len(missing_names),
        "unexpectedCount": len(unexpected_names),
        "missingExamples": missing_names[:10],
        "unexpectedExamples": unexpected_names[:10],
    })

print({
    "stage": "copyingArchiveToDrive",
    "archiveFiles": len(actual_names),
})

# Copiar el único archivo grande a Drive con progreso.
total_bytes = LOCAL_ARCHIVE.stat().st_size
copied_bytes = 0
next_report = 512 * 1024**2

with (
    LOCAL_ARCHIVE.open("rb") as source,
    DRIVE_ARCHIVE_PARTIAL.open("wb") as target,
):
    while True:
        chunk = source.read(
            64 * 1024**2
        )

        if not chunk:
            break

        target.write(chunk)
        copied_bytes += len(chunk)

        if (
            copied_bytes >= next_report
            or copied_bytes == total_bytes
        ):
            print({
                "copiedGiB": round(
                    copied_bytes / 1024**3,
                    2,
                ),
                "totalGiB": round(
                    total_bytes / 1024**3,
                    2,
                ),
                "percent": round(
                    100
                    * copied_bytes
                    / total_bytes,
                    2,
                ),
            })

            next_report += 512 * 1024**2

os.replace(
    DRIVE_ARCHIVE_PARTIAL,
    DRIVE_ARCHIVE,
)

print(json.dumps({
    "archive": str(DRIVE_ARCHIVE),
    "sizeGiB": round(
        DRIVE_ARCHIVE.stat().st_size
        / 1024**3,
        2,
    ),
    "trainFiles": len(train_files),
    "validationFiles": len(validation_files),
    "archiveFiles": len(actual_names),
    "ready": True,
}, indent=2))

{'train': 13445, 'validation': 2894, 'total': 16339}
{'localFreeGiB': 201.14}
{'archived': 500, 'total': 16339, 'percent': 3.06, 'minutes': 1.8}
{'archived': 1000, 'total': 16339, 'percent': 6.12, 'minutes': 1.8}
{'archived': 1500, 'total': 16339, 'percent': 9.18, 'minutes': 1.8}
{'archived': 2000, 'total': 16339, 'percent': 12.24, 'minutes': 1.9}
{'archived': 2500, 'total': 16339, 'percent': 15.3, 'minutes': 1.9}
{'archived': 3000, 'total': 16339, 'percent': 18.36, 'minutes': 2.0}
{'archived': 3500, 'total': 16339, 'percent': 21.42, 'minutes': 2.0}
{'archived': 4000, 'total': 16339, 'percent': 24.48, 'minutes': 2.0}
{'archived': 4500, 'total': 16339, 'percent': 27.54, 'minutes': 2.1}
{'archived': 5000, 'total': 16339, 'percent': 30.6, 'minutes': 2.1}
{'archived': 5500, 'total': 16339, 'percent': 33.66, 'minutes': 2.2}
{'archived': 6000, 'total': 16339, 'percent': 36.72, 'minutes': 2.2}
{'archived': 6500, 'total': 16339, 'percent': 39.78, 'minutes': 2.3}
{'archived': 7000, 'total': 163

## 3B — GPU: copiar el caché al SSD local

Después de completar 3A, cambiar a T4 o superior, volver a ejecutar 1 y 2, y luego ejecutar 3B.


In [5]:
# 3B) Validar caché persistente y copiarlo al SSD local
if not torch.cuda.is_available():
    raise RuntimeError("3B requiere GPU T4 o superior.")

train_samples = prepare_samples(train_manifest, data_root, "train")
validation_samples = prepare_samples(
    validation_manifest,
    data_root,
    "validation",
)

def audit_exact_cache(samples, root: Path, split: str) -> dict:
    expected = set(samples["cache_file"].astype(str))
    actual = {path.name for path in (root / split).glob("*.npy")}
    return {
        "expected": len(expected),
        "present": len(actual),
        "missing": sorted(expected - actual)[:20],
        "unexpected": sorted(actual - expected)[:20],
        "complete": expected == actual,
    }

persistent_train = audit_exact_cache(
    train_samples, PERSISTENT_CACHE_ROOT, "train"
)
persistent_validation = audit_exact_cache(
    validation_samples, PERSISTENT_CACHE_ROOT, "validation"
)

if not (
    persistent_train["complete"]
    and persistent_validation["complete"]
):
    raise RuntimeError({
        "message": "El caché persistente está incompleto.",
        "train": persistent_train,
        "validation": persistent_validation,
    })

if LOCAL_CACHE_ROOT.exists():
    shutil.rmtree(LOCAL_CACHE_ROOT)
LOCAL_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

subprocess.run(
    [
        "rsync", "-a", "--delete", "--info=progress2",
        f"{PERSISTENT_CACHE_ROOT}/",
        f"{LOCAL_CACHE_ROOT}/",
    ],
    check=True,
)

local_train = audit_exact_cache(train_samples, LOCAL_CACHE_ROOT, "train")
local_validation = audit_exact_cache(
    validation_samples,
    LOCAL_CACHE_ROOT,
    "validation",
)
if not (local_train["complete"] and local_validation["complete"]):
    raise RuntimeError("La copia local del caché quedó incompleta.")

CACHE_ROOT = LOCAL_CACHE_ROOT
print({
    "gpu": torch.cuda.get_device_name(0),
    "trainingCacheRoot": str(CACHE_ROOT),
    "trainFiles": local_train["present"],
    "validationFiles": local_validation["present"],
    "readyForTraining": True,
})


KeyboardInterrupt: 

## 4 — Entrenamiento

El mejor checkpoint se selecciona exclusivamente con `validation`. La implementación guarda checkpoints por epoch, pero no reanuda automáticamente el optimizador ni el epoch si esta celda se interrumpe.


In [3]:
# 4) Entrenar y evaluar sobre validation
if not torch.cuda.is_available():
    raise RuntimeError("La celda 4 requiere GPU.")
if "CACHE_ROOT" not in globals() or CACHE_ROOT != LOCAL_CACHE_ROOT:
    raise RuntimeError("Ejecutá 3B antes de entrenar.")

summary = train_model(
    train_samples=train_samples,
    validation_samples=validation_samples,
    cache_root=CACHE_ROOT,
    checkpoint_root=CHECKPOINT_ROOT,
    run_root=RUN_ROOT,
    manifest_hashes=manifest_hashes,
    repo_ref=REPO_REF,
    repo_sha=REPO_SHA,
    config=CFG,
)
print(json.dumps(summary, indent=2, ensure_ascii=False))


model.safetensors: reconstructing file:   0%|          |  0.00B / 21.4MB            

model.safetensors: downloading bytes:           |  0.00B            

train:   0%|          | 0/421 [00:00<?, ?it/s]

validation:   0%|          | 0/91 [00:00<?, ?it/s]

{'epoch': 1, 'learning_rate': 0.0002, 'train_macro_f1': 0.6482530505848202, 'train_balanced_accuracy': 0.6718816784057786, 'train_normal_mild_recall': 0.6138681169272604, 'train_moderate_recall': 0.5361271676300579, 'train_severe_recall': 0.8656497506600176, 'train_weighted_log_loss': 0.8083156771049644, 'train_selection_score': 0.697296794263383, 'train_loss': 0.6937527179851014, 'validation_macro_f1': 0.5818944728356791, 'validation_balanced_accuracy': 0.6582718246122754, 'validation_normal_mild_recall': 0.7407407407407407, 'validation_moderate_recall': 0.45907473309608543, 'validation_severe_recall': 0.775, 'validation_weighted_log_loss': 0.7502521939167096, 'validation_selection_score': 0.6369832185969491, 'validation_loss': 0.9380389564846331}


...

train:   0%|          | 0/421 [00:00<?, ?it/s]

validation:   0%|          | 0/91 [00:00<?, ?it/s]

{'epoch': 11, 'learning_rate': 2.5e-05, 'train_macro_f1': 0.9806350392836273, 'train_balanced_accuracy': 0.983448300787721, 'train_normal_mild_recall': 0.9629443326626426, 'train_moderate_recall': 0.9892138063279002, 'train_severe_recall': 0.9981867633726201, 'train_weighted_log_loss': 0.13596649329860158, 'train_selection_score': 0.9865841623863262, 'train_loss': 0.21420989568546475, 'validation_macro_f1': 0.6564609496511067, 'validation_balanced_accuracy': 0.6551576235302901, 'validation_normal_mild_recall': 0.8869395711500975, 'validation_moderate_recall': 0.5106761565836299, 'validation_severe_recall': 0.5678571428571428, 'validation_weighted_log_loss': 0.6383157307667164, 'validation_selection_score': 0.6194056871156639, 'validation_loss': 1.0058106643295157}
{'earlyStopping': True, 'epoch': 11}


validation:   0%|          | 0/91 [00:00<?, ?it/s]

{
  "schemaVersion": "pfi.rsna-subarticular-training.v1",
  "ticket": "P10.6-AI",
  "notebook": 63,
  "sourceNotebook": 62,
  "status": "APPROVED_FOR_NOTEBOOK_64",
  "approved": true,
  "nextNotebook": 64,
  "createdAtUtc": "2026-08-06T00:42:40.650155+00:00",
  "task": "subarticular_stenosis_left_right",
  "sequence": "Axial T2",
  "architecture": {
    "backbone": "efficientnet_b0",
    "input": "2.5D_three_adjacent_axial_slices",
    "sharedSides": true,
    "sideEmbedding": 8,
    "levelEmbedding": 12,
    "pretrainedLoaded": true
  },
  "config": {
    "seed": 2026,
    "image_size": 224,
    "crop_size": 256,
    "batch_size": 32,
    "num_workers": 2,
    "max_epochs": 15,
    "patience": 5,
    "learning_rate": 0.0002,
    "weight_decay": 0.0001,
    "model_name": "efficientnet_b0",
    "pretrained": true,
    "side_embedding_dim": 8,
    "level_embedding_dim": 12,
    "dropout": 0.25,
    "label_smoothing": 0.05,
    "severe_loss_multiplier": 1.25,
    "max_grad_norm": 2.0,
   

In [4]:
# 5) Gate final y artefactos
required_outputs = [
    "training_history.csv",
    "validation_predictions.csv",
    "validation_metrics_by_group.csv",
    "sampling_audit.json",
    "model_card.md",
    "training_summary.json",
]
missing_outputs = [
    name
    for name in required_outputs
    if not (RUN_ROOT / name).is_file()
]
if missing_outputs:
    raise RuntimeError(f"Faltan outputs: {missing_outputs}")
if summary["approved"] is not True:
    raise RuntimeError(
        "El entrenamiento requiere revisión. "
        "No abrir el internal test ni ajustar gates con él."
    )
if summary["status"] != "APPROVED_FOR_NOTEBOOK_64":
    raise RuntimeError("Estado final inesperado.")
if summary["governance"]["internalTestAccessed"] is not False:
    raise RuntimeError("Se declaró acceso indebido al internal test.")

print({
    "status": summary["status"],
    "bestEpoch": summary["bestEpoch"],
    "validationMetrics": summary["validationMetrics"],
    "checkpoint": summary["checkpoint"],
    "outputs": required_outputs,
    "internalTestSealedUntilNotebook64": True,
    "officialTestAccessed": False,
})


{'status': 'APPROVED_FOR_NOTEBOOK_64', 'bestEpoch': 6, 'validationMetrics': {'macro_f1': 0.6465925601604275, 'balanced_accuracy': 0.6785678526750513, 'normal_mild_recall': 0.8411306042884991, 'moderate_recall': 0.5195729537366548, 'severe_recall': 0.675, 'weighted_log_loss': 0.6833057247019069, 'selection_score': 0.6489862826065994, 'loss': 0.9799525498437651}, 'checkpoint': {'path': '/content/drive/MyDrive/PFI_MVP/models/P10_6_rsna_findings/subarticular_axial_t2_2p5d/checkpoints/best_checkpoint.pt', 'sha256': 'd41262d57b13c146a48ab15f5e183cc6a55fc92724b7d0c286cea1f2ce26e84a'}, 'outputs': ['training_history.csv', 'validation_predictions.csv', 'validation_metrics_by_group.csv', 'sampling_audit.json', 'model_card.md', 'training_summary.json'], 'internalTestSealedUntilNotebook64': True, 'officialTestAccessed': False}
